# Первый запуск ThermoMPNN

Цель: получить прогнозы для нескольких мутаций одного белка
из обучающей выборки и сравнить их с экспериментом.

Перед предсказанием проверяем окружение, соответствие позиций
и соглашение о знаке изменения стабильности.

In [1]:
from pathlib import Path
import sys
import torch

# Тетрадка может запускаться из корня проекта или из notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

THERMOMPNN_DIR = PROJECT_ROOT / "external" / "ThermoMPNN"

print("Python:", sys.version.split()[0])
print("Интерпретатор:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
print("Корень проекта:", PROJECT_ROOT)

assert (PROJECT_ROOT / "pyproject.toml").is_file(), \
    "Не найден pyproject.toml — проверь расположение тетрадки."

assert (THERMOMPNN_DIR / "models" / "thermoMPNN_default.pt").is_file(), \
    "Не найдены веса ThermoMPNN — проверь клонирование репозитория."

print("Окружение и файл весов найдены.")

Python: 3.10.21
Интерпретатор: /home/andre/miniconda3/envs/NeuroPP/bin/python
PyTorch: 2.5.1+cpu
CUDA доступна: False
Корень проекта: /home/andre/Neuro_Project_Pilot
Окружение и файл весов найдены.


## Загрузка модели

Создаём ThermoMPNN с конфигурацией авторов и загружаем обученные
веса. Используем CPU и переводим модель в режим предсказания.

In [2]:
# Позволяем Python импортировать код из репозитория ThermoMPNN.
if str(THERMOMPNN_DIR) not in sys.path:
    sys.path.insert(0, str(THERMOMPNN_DIR))

from omegaconf import OmegaConf
from train_thermompnn import TransferModelPL

DEVICE = torch.device("cpu")

# Читаем конфигурацию авторов и указываем наши локальные пути.
cfg = OmegaConf.load(THERMOMPNN_DIR / "config.yaml")
cfg.platform = {
    "thermompnn_dir": str(THERMOMPNN_DIR),
    "accel": "cpu",
}

/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [3]:
checkpoint_path = THERMOMPNN_DIR / "models" / "thermoMPNN_default.pt"

model = TransferModelPL.load_from_checkpoint(
    str(checkpoint_path),
    cfg=cfg,
    map_location=DEVICE,
    strict=True,
)

model.eval()

print("Устройство модели:", next(model.parameters()).device)
print("Режим обучения:", model.training)
print("Количество параметров:", sum(p.numel() for p in model.parameters()))

/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/home/andre/Neuro_Project_Pilot/exte

Устройство модели: cpu
Режим обучения: False
Количество параметров: 4342876


/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


## Выбор белка

Используем официальное разбиение MegaScale на train, validation и test.
Для первого запуска выбираем один белок из train.
Это проверка работы тетрадки, а не оценка обобщающей способности модели.

In [4]:
import pickle

splits_path = THERMOMPNN_DIR / "dataset_splits" / "mega_splits.pkl"

with splits_path.open("rb") as file:
    splits = pickle.load(file)

for split_name in ("train", "val", "test"):
    print(f"{split_name}: {len(splits[split_name])} белков")

# Выбираем первый идентификатор по алфавиту для воспроизводимости.
protein_id = sorted(splits["train"])[0]

assert protein_id not in splits["val"]
assert protein_id not in splits["test"]

print("Выбранный белок:", protein_id)

train: 239 белков
val: 31 белков
test: 28 белков
Выбранный белок: 1A32.pdb


## Получение данных MegaScale

Источник: https://zenodo.org/records/7992926

Скачиваем экспериментальные данные и структуры AlphaFold.
Проверяем контрольные суммы архивов перед использованием.

In [5]:
import subprocess

DATA_DIR = PROJECT_ROOT / "data" / "megascale"
DATA_DIR.mkdir(parents=True, exist_ok=True)

archive_names = [
    "Processed_K50_dG_datasets.zip",
    "AlphaFold_model_PDBs.zip",
]

for name in archive_names:
    url = f"https://zenodo.org/api/records/7992926/files/{name}/content"
    archive_path = DATA_DIR / name

    print(f"Скачиваем {name}", flush=True)

    subprocess.run(
        [
            "wget", "--continue", "--progress=dot:giga",
            "--output-document", str(archive_path), url,
        ],
        check=True,
    )

Скачиваем Processed_K50_dG_datasets.zip


--2026-09-29 16:42:53--  https://zenodo.org/api/records/7992926/files/Processed_K50_dG_datasets.zip/content
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.185.48.75, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.


Скачиваем AlphaFold_model_PDBs.zip


HTTP request sent, awaiting response... 416 REQUESTED_RANGE_NOT_SATISFIABLE

    The file is already fully retrieved; nothing to do.

--2026-09-29 16:42:55--  https://zenodo.org/api/records/7992926/files/AlphaFold_model_PDBs.zip/content
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.185.48.75, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 416 REQUESTED_RANGE_NOT_SATISFIABLE

    The file is already fully retrieved; nothing to do.



In [6]:
import hashlib

expected_md5 = {
    "Processed_K50_dG_datasets.zip": "f7e8c553efee734cf161ee6f2b0a09cf",
    "AlphaFold_model_PDBs.zip": "c29317a75f28d008b2718316995902ef",
}

for name, expected in expected_md5.items():
    digest = hashlib.md5()

    with (DATA_DIR / name).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    assert digest.hexdigest() == expected, f"Не совпала контрольная сумма: {name}"
    print(f"{name}: OK")

Processed_K50_dG_datasets.zip: OK
AlphaFold_model_PDBs.zip: OK


## Чтение данных выбранного белка

Извлекаем таблицу Dataset2_Dataset3 и структуру выбранного белка.
Читаем таблицу частями, оставляя записи с нужным WT_name.

In [7]:
import zipfile

csv_name = "Tsuboyama2023_Dataset2_Dataset3_20230416.csv"

with zipfile.ZipFile(DATA_DIR / "Processed_K50_dG_datasets.zip") as archive:
    csv_path = Path(
        archive.extract(
            f"Processed_K50_dG_datasets/{csv_name}",
            path=DATA_DIR,
        )
    )

with zipfile.ZipFile(DATA_DIR / "AlphaFold_model_PDBs.zip") as archive:
    pdb_path = Path(
        archive.extract(
            f"AlphaFold_model_PDBs/{protein_id}",
            path=DATA_DIR,
        )
    )

print("Таблица:", csv_path)
print("Структура:", pdb_path)

Таблица: /home/andre/Neuro_Project_Pilot/data/megascale/Processed_K50_dG_datasets/Tsuboyama2023_Dataset2_Dataset3_20230416.csv
Структура: /home/andre/Neuro_Project_Pilot/data/megascale/AlphaFold_model_PDBs/1A32.pdb


In [8]:
import pandas as pd

columns = ["WT_name", "mut_type", "aa_seq", "dG_ML", "ddG_ML"]
selected_chunks = []

with pd.read_csv(
    csv_path,
    usecols=columns,
    dtype=str,
    keep_default_na=False,
    chunksize=50_000,
) as reader:
    for chunk in reader:
        selected = chunk.loc[chunk["WT_name"] == protein_id].copy()

        if not selected.empty:
            selected_chunks.append(selected)

assert selected_chunks, f"В таблице не найден белок {protein_id}"

protein_rows = pd.concat(selected_chunks, ignore_index=True)

print("Всего записей для белка:", len(protein_rows))
print("Записей исходного варианта WT:", (protein_rows["mut_type"] == "wt").sum())

display(protein_rows[["mut_type", "dG_ML", "ddG_ML"]].head(8))

Всего записей для белка: 1391
Записей исходного варианта WT: 5


,mut_type,dG_ML,ddG_ML
0,wt,2.674475783114864,0.10354545447474273
1,wt,2.6442046890894666,0.07327436044934554
2,wt,2.570930328640121,0.0
3,wt,2.564956081208039,-0.00597424743208208
4,wt,2.533722534035041,-0.03720779460507995
5,S1Q,2.343605008511613,-0.22732532012850815
6,S1E,2.469427860637595,-0.10150246800252605
7,S1N,2.6136095476739083,0.04267921903378724


## Проверка исходного белка

Проверяем последовательности записей WT и сравниваем их
с последовательностью, прочитанной из PDB.

Проверяем опорное значение энергии, использованное в ddG_ML.

In [9]:
protein_data = protein_rows.copy()

# Преобразуем числовые значения; отсутствующие оценки станут NaN.
for column in ("dG_ML", "ddG_ML"):
    protein_data[column] = pd.to_numeric(
        protein_data[column], errors="coerce"
    )

wt_rows = protein_data.loc[protein_data["mut_type"] == "wt"].copy()

wt_rows["sequence_length"] = wt_rows["aa_seq"].str.len()

# Восстанавливаем опорное значение из определения ddG_ML.
wt_rows["reference_dG"] = wt_rows["dG_ML"] - wt_rows["ddG_ML"]

print("Уникальных последовательностей WT:", wt_rows["aa_seq"].nunique())

display(
    wt_rows[
        ["sequence_length", "dG_ML", "ddG_ML", "reference_dG"]
    ].round(6)
)

Уникальных последовательностей WT: 1


,sequence_length,dG_ML,ddG_ML,reference_dG
0,63,2.674476,0.103545,2.57093
1,63,2.644205,0.073274,2.57093
2,63,2.570930,0.000000,2.57093
3,63,2.564956,-0.005974,2.57093
4,63,2.533723,-0.037208,2.57093


In [10]:
from protein_mpnn_utils import parse_PDB

pdb_batch = parse_PDB(str(pdb_path))

assert len(pdb_batch) == 1, "Ожидалась одна запись структуры."

pdb_record = pdb_batch[0]
pdb_sequence = pdb_record["seq"]

print("Число цепей:", pdb_record["num_of_chains"])
print("Длина последовательности PDB:", len(pdb_sequence))
print("Последовательность PDB:", pdb_sequence)

wt_rows["matches_pdb"] = wt_rows["aa_seq"] == pdb_sequence

display(wt_rows[["sequence_length", "matches_pdb"]])

Число цепей: 1
Длина последовательности PDB: 63
Последовательность PDB: SPEVQIAILTEQINNLNEHLRVHKKDHHSRRGLLKMVGKRRRLLAYLRNKDVARYREIVEKLG


,sequence_length,matches_pdb
0,63,True
1,63,True
2,63,True
3,63,True
4,63,True


## Подготовка мутаций

Оставляем одиночные аминокислотные замены с числовой оценкой эффекта.
Проверяем позиции и последовательности мутантов.

Цель: ddg_exp = -ddG_ML; положительное значение — дестабилизация.

In [11]:
import numpy as np

# Однобуквенные обозначения 20 стандартных аминокислот.
amino_acids = "ACDEFGHIKLMNPQRSTVWY"

# Формат: аминокислота → положительный номер позиции → аминокислота.
pattern = rf"[{amino_acids}][1-9][0-9]*[{amino_acids}]"
is_single = protein_data["mut_type"].str.fullmatch(pattern, na=False)

single_mutants = protein_data.loc[
    is_single & np.isfinite(protein_data["ddG_ML"])
].copy()

single_mutants["wt_aa"] = single_mutants["mut_type"].str[0]
single_mutants["mut_aa"] = single_mutants["mut_type"].str[-1]
single_mutants["position"] = (
    single_mutants["mut_type"].str[1:-1].astype(int)
)
single_mutants["index0"] = single_mutants["position"] - 1

# Исключаем обозначения, в которых аминокислота фактически не меняется.
single_mutants = single_mutants.loc[
    single_mutants["wt_aa"] != single_mutants["mut_aa"]
].copy()

single_mutants["ddg_exp"] = -single_mutants["ddG_ML"]

print("Одиночных замен с числовой оценкой:", len(single_mutants))

Одиночных замен с числовой оценкой: 1195


In [12]:
assert wt_rows["aa_seq"].nunique() == 1
assert pdb_record["num_of_chains"] == 1

wt_sequence = wt_rows["aa_seq"].iloc[0]
assert wt_sequence == pdb_sequence

for row in single_mutants.itertuples():
    i = row.index0

    assert 0 <= i < len(wt_sequence), f"Позиция вне белка: {row.mut_type}"
    assert wt_sequence[i] == row.wt_aa, f"Неверная WT-буква: {row.mut_type}"

    expected_sequence = wt_sequence[:i] + row.mut_aa + wt_sequence[i + 1:]
    assert row.aa_seq == expected_sequence, \
        f"Последовательность не соответствует замене: {row.mut_type}"

assert len(single_mutants) >= 5
examples = single_mutants.sample(n=5, random_state=42).copy()

print("Позиции и последовательности прошли проверку.")

display(
    examples[
        ["mut_type", "position", "index0", "wt_aa", "mut_aa", "ddg_exp"]
    ]
)

Позиции и последовательности прошли проверку.


,mut_type,position,index0,wt_aa,mut_aa,ddg_exp
740,L34A,34,33,L,A,-0.050672
284,I13Y,13,12,I,Y,1.467750
1264,I58T,58,57,I,T,0.675720
1291,V59L,59,58,V,L,-0.261913
72,V4E,4,3,V,E,-0.360069


## Предсказания ThermoMPNN

Передаём модели структуру WT и описания пяти мутаций.
Экспериментальные значения используются только для сравнения.

Все эффекты и ошибки выражены в ккал/моль.
Положительный эффект означает дестабилизацию.

In [13]:
from datasets import Mutation

# Проверяем, что координаты атомов не содержат пропусков.
coords_key = next(
    key for key in pdb_record if key.startswith("coords_chain_")
)
assert all(
    np.isfinite(xyz).all() for xyz in pdb_record[coords_key].values()
), "В структуре есть отсутствующие или некорректные координаты."

# Преобразуем строки таблицы в описания мутаций для модели.
mutations = [
    Mutation(
        position=int(row.index0),
        wildtype=row.wt_aa,
        mutation=row.mut_aa,
    )
    for row in examples.itertuples()
]

model.eval()

with torch.inference_mode():
    predictions, _ = model(pdb_batch, mutations)

assert len(predictions) == len(examples)

print("Получено прогнозов:", len(predictions))
print("Первый результат модели:", predictions[0]["ddG"])

Получено прогнозов: 5
Первый результат модели: tensor([-0.0955])


In [14]:
results = examples[
    ["mut_type", "position", "index0", "ddg_exp"]
].copy().reset_index(drop=True)

# .item() превращает тензор с одним элементом в обычное число.
results["ddg_pred"] = [
    prediction["ddG"].item() for prediction in predictions
]

assert np.isfinite(results["ddg_pred"]).all(), "Получены некорректные прогнозы."

results["error"] = results["ddg_pred"] - results["ddg_exp"]
results["abs_error"] = results["error"].abs()

display(results.round(4))

print(
    "MAE на пяти примерах:",
    round(results["abs_error"].mean(), 4),
    "ккал/моль",
)

,mut_type,position,index0,ddg_exp,ddg_pred,error,abs_error
0,L34A,34,33,-0.0507,-0.0955,-0.0448,0.0448
1,I13Y,13,12,1.4677,1.0422,-0.4255,0.4255
2,I58T,58,57,0.6757,0.8818,0.2061,0.2061
3,V59L,59,58,-0.2619,0.1746,0.4365,0.4365
4,V4E,4,3,-0.3601,-0.2813,0.0788,0.0788


MAE на пяти примерах: 0.2383 ккал/моль


In [15]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / f"{Path(protein_id).stem}_first_predictions.csv"

if results_path.exists():
    # При повторном запуске сравниваем с ранее сохранённым результатом.
    saved_results = pd.read_csv(results_path)

    pd.testing.assert_frame_equal(
        results,
        saved_results,
        check_dtype=False,
        check_exact=False,
        rtol=1e-6,
        atol=1e-6,
    )

    print("Повторный результат совпал с сохранённым.")
else:
    results.to_csv(results_path, index=False)
    print("Первый результат сохранён:", results_path)

Повторный результат совпал с сохранённым.


In [16]:
sources_path = PROJECT_ROOT / "sources.yaml"
sources = OmegaConf.load(sources_path)

sources.data = [
    {
        "name": name,
        "url": f"https://zenodo.org/api/records/7992926/files/{name}/content",
        "md5": checksum,
    }
    for name, checksum in expected_md5.items()
]

OmegaConf.save(sources, sources_path)

print("Источники данных записаны в sources.yaml.")

Источники данных записаны в sources.yaml.


## Результат первого запуска

Проверены последовательность WT, соответствие структуре,
позиции мутаций и соглашение о знаке эффекта.

Получены прогнозы для пяти мутаций белка 1A32 из train.
MAE на этих примерах: 0,2383 ккал/моль.
Для V59L знак прогноза отличается от экспериментального.

Этот запуск проверяет работу процесса и не оценивает качество
модели на новых белках.